# Stage 1 — Exploratory Data Analysis (EDA)

**Goal:** Understand the data before building any model. Never touch a model until you know what the data looks like.

In a real telecom company, data comes from 3 separate systems:
- **CRM system** → customer demographics (tenure, partner, dependents)
- **Provisioning system** → service subscriptions (internet, phone, add-ons)
- **Billing system** → contract type, charges, churn outcome
- **Network system** → CDR events (calls, data, SMS, complaints)

We unified all 4 into the `analytics.customer_360` view via SQL.  
This notebook reads directly from that view — not from flat files.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from db import query

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

print('Libraries loaded')

## 1. Load Customer 360 from PostgreSQL

Instead of `pd.read_csv()`, we query the warehouse view.  
This is how real analysts work — the warehouse is the single source of truth.

In [ ]:
df = query('SELECT * FROM analytics.customer_360')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

## 2. Dataset Overview

In [ ]:
print('--- Data types ---')
print(df.dtypes)
print('\n--- Missing values ---')
nulls = df.isnull().sum()
print(nulls[nulls > 0])

In [ ]:
# churn_reason is only populated for churned customers — that's expected
df[['tenure', 'monthly_charges', 'total_charges', 'churn_score', 'cltv',
    'total_calls', 'total_call_mins', 'total_data_mb', 'complaint_count']].describe().round(2)

## 3. Churn Rate — The Key Business Metric

Before any modelling, understand the base rate.  
A high churn rate means class imbalance — our model must handle this.

In [ ]:
churn_counts = df['churn'].value_counts()
churn_rate   = (df['churn'] == 'Yes').mean()
revenue_at_risk = df[df['churn'] == 'Yes']['monthly_charges'].sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
bars = axes[0].bar(['Retained', 'Churned'], churn_counts.values,
                   color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Customer Count by Churn Status', fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for bar, val in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 40,
                 f'{val:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
            startangle=90, explode=[0, 0.05])
axes[1].set_title(f'Churn Rate: {churn_rate:.1%}', fontweight='bold')

plt.suptitle('Overall Churn Distribution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Overall churn rate  : {churn_rate:.2%}')
print(f'Monthly revenue at risk: ${revenue_at_risk:,.0f}')

## 4. Numerical Feature Distributions

Look at the shape of each numeric feature — are they skewed? Any outliers?  
This tells us whether we need to scale or transform them later.

In [ ]:
num_cols = ['tenure', 'monthly_charges', 'total_charges', 'churn_score', 'cltv']

fig, axes = plt.subplots(2, len(num_cols), figsize=(18, 8))

for i, col in enumerate(num_cols):
    # Distribution
    sns.histplot(df[col], ax=axes[0, i], kde=True, color='steelblue')
    axes[0, i].set_title(col, fontweight='bold')
    axes[0, i].set_xlabel('')

    # Split by churn
    for label, color in [('No', '#2ecc71'), ('Yes', '#e74c3c')]:
        sns.kdeplot(df[df['churn'] == label][col], ax=axes[1, i],
                    label=f'Churn={label}', color=color, fill=True, alpha=0.3)
    axes[1, i].set_title(f'{col} by Churn', fontweight='bold')
    axes[1, i].legend(fontsize=8)

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Churn Rate by Contract Type

**Business insight:** Contract type is usually the single strongest predictor of churn.  
Month-to-month customers have no lock-in — they can leave any time.

In [ ]:
contract_churn = (df.groupby('contract')['churn']
                    .apply(lambda x: (x == 'Yes').mean())
                    .reset_index()
                    .rename(columns={'churn': 'churn_rate'})
                    .sort_values('churn_rate', ascending=False))

colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = plt.bar(contract_churn['contract'], contract_churn['churn_rate'],
               color=colors, edgecolor='white')
plt.title('Churn Rate by Contract Type', fontweight='bold', fontsize=13)
plt.ylabel('Churn Rate')
plt.ylim(0, 0.60)
for bar, val in zip(bars, contract_churn['churn_rate']):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.01,
             f'{val:.1%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Churn Rate by Internet Service Type

In [ ]:
internet_churn = (df.groupby('internet_service')['churn']
                    .apply(lambda x: (x == 'Yes').mean())
                    .reset_index()
                    .rename(columns={'churn': 'churn_rate'})
                    .sort_values('churn_rate', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Churn rate
axes[0].bar(internet_churn['internet_service'], internet_churn['churn_rate'],
            color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white')
axes[0].set_title('Churn Rate by Internet Service', fontweight='bold')
axes[0].set_ylabel('Churn Rate')
for i, (_, row) in enumerate(internet_churn.iterrows()):
    axes[0].text(i, row['churn_rate'] + 0.01, f"{row['churn_rate']:.1%}",
                 ha='center', fontweight='bold')

# Monthly charges by internet service and churn
sns.boxplot(data=df, x='internet_service', y='monthly_charges', hue='churn',
            palette={'No': '#2ecc71', 'Yes': '#e74c3c'}, ax=axes[1])
axes[1].set_title('Monthly Charges by Internet Service & Churn', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Churn Rate by Tenure Segment

**Business insight:** New customers churn the most.  
The first 12 months is the critical retention window.

In [ ]:
segment_order = ['New', 'Growing', 'Loyal', 'Champion']

seg_churn = (df.groupby('tenure_segment')['churn']
               .apply(lambda x: (x == 'Yes').mean())
               .reindex(segment_order)
               .reset_index()
               .rename(columns={'churn': 'churn_rate'}))

seg_count = df.groupby('tenure_segment').size().reindex(segment_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(seg_churn['tenure_segment'], seg_churn['churn_rate'],
            color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'], edgecolor='white')
axes[0].set_title('Churn Rate by Tenure Segment', fontweight='bold')
axes[0].set_ylabel('Churn Rate')
for i, val in enumerate(seg_churn['churn_rate']):
    axes[0].text(i, val + 0.005, f'{val:.1%}', ha='center', fontweight='bold')

axes[1].bar(seg_count.index, seg_count.values,
            color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'], edgecolor='white')
axes[1].set_title('Customer Count by Tenure Segment', fontweight='bold')
axes[1].set_ylabel('Customers')
for i, val in enumerate(seg_count.values):
    axes[1].text(i, val + 20, f'{val:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. CDR Analysis — Usage vs Churn

This is unique to our project — we have actual network usage data.  
Churned customers typically show **declining usage** before they leave.

In [ ]:
cdr_cols = ['total_calls', 'total_call_mins', 'total_data_mb', 'complaint_count']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(cdr_cols):
    sns.boxplot(data=df, x='churn', y=col,
                palette={'No': '#2ecc71', 'Yes': '#e74c3c'}, ax=axes[i])
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Churn')

plt.suptitle('CDR Usage Metrics by Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Mean comparison
print('Mean CDR metrics by churn status:')
print(df.groupby('churn')[cdr_cols].mean().round(1))

## 9. Top Churn Reasons (from IBM dataset)

This column only exists in the richer IBM version. In a real project, this data  
would come from call centre notes or exit surveys.

In [ ]:
reasons = (df[df['churn'] == 'Yes']['churn_reason']
             .value_counts()
             .head(10))

plt.figure(figsize=(12, 5))
bars = plt.barh(reasons.index[::-1], reasons.values[::-1], color='#e74c3c', edgecolor='white')
plt.title('Top 10 Churn Reasons', fontweight='bold', fontsize=13)
plt.xlabel('Number of Customers')
for bar, val in zip(bars, reasons.values[::-1]):
    plt.text(val + 5, bar.get_y() + bar.get_height()/2, str(val), va='center')
plt.tight_layout()
plt.show()

## 10. Correlation Heatmap

In [ ]:
corr_cols = ['tenure', 'monthly_charges', 'total_charges', 'churn_score',
             'cltv', 'num_services', 'total_calls', 'total_call_mins',
             'total_data_mb', 'complaint_count', 'churn_value']

plt.figure(figsize=(13, 10))
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix — Numerical Features', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Key Business Insights

Run the cell below to print a summary of what we found — these are the talking points  
you would present to stakeholders before building the model.

In [ ]:
mtm_churn  = (df[df['contract'] == 'Month-to-month']['churn'] == 'Yes').mean()
new_churn  = (df[df['tenure_segment'] == 'New']['churn'] == 'Yes').mean()
fiber_churn = (df[df['internet_service'] == 'Fiber optic']['churn'] == 'Yes').mean()
avg_complaints_churned = df[df['churn'] == 'Yes']['complaint_count'].mean()
avg_complaints_retained = df[df['churn'] == 'No']['complaint_count'].mean()

print('=' * 55)
print('  KEY BUSINESS INSIGHTS FROM EDA')
print('=' * 55)
print(f'  Overall churn rate         : {churn_rate:.1%}')
print(f'  Month-to-month churn rate  : {mtm_churn:.1%}  ← highest risk')
print(f'  New customer churn rate    : {new_churn:.1%}  ← first 12 months critical')
print(f'  Fiber optic churn rate     : {fiber_churn:.1%}  ← high charges, low loyalty')
print(f'  Avg complaints (churned)   : {avg_complaints_churned:.1f}')
print(f'  Avg complaints (retained)  : {avg_complaints_retained:.1f}')
print(f'  Monthly revenue at risk    : ${revenue_at_risk:,.0f}')
print('=' * 55)
print()
print('  WHAT TO DO NEXT:')
print('  → Engineer features from these insights')
print('  → Target Month-to-month customers with retention offers')
print('  → Flag new customers in first 12 months for proactive outreach')
print('  → Use complaint_count as a churn predictor in the model')